<a href="https://colab.research.google.com/github/MuhammadEhtisham776/flyrank-ml-internship-starter/blob/main/Copy_of_w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadEhtisham776/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [8]:
from google.colab import userdata
import duckdb

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(
    f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

rel = "hf://datasets/FlyRank/internship-warehouse"

In [9]:
con.sql(f"""
SELECT COUNT(*)
FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet'
)
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│     78835655 │
└──────────────┘

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**One row = one page-month**: a `(client_hash_id, content_hash_id)` pair, with clicks and
impressions **summed** across every daily row in the month (not one row per day, and not an
average of daily averages — `avg_position` is rebuilt from `sum(gsc_sum_position) / sum(gsc_impressions)`,
never from averaging the daily `gsc_avg_position` column).

**Table(s):** `fact_content_daily_performance/month=2026-03` (the daily grain) aggregated up to
page-month, joined to `dim_content` for `content_type`, `main_intent`, `word_count`, `search_volume`.

**Time window:** 2026-03-01 to 2026-03-31 — a mid-panel month, per the panel warning (never the
`_sample` table, which is the sealed final month).

**Target/proxy:** `ctr_gap` = a page's monthly CTR minus the median CTR of other visible pages in
the same position tier that same month — same construction as Weeks 1-2, now on real warehouse data.

**Deliberately excluded:** GA4 columns (`ga4_*`, `sessions_*`, `ai_*`) — not needed for this lane
and mostly zero-filled/unavailable for clients without GA4 access; raw `report_date`-level rows —
collapsed to page-month since Lane 4 works at that grain, not page-day.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd
import numpy as np

REPO = "hf://datasets/FlyRank/internship-warehouse"

fact = pd.read_parquet(f"{REPO}/fact_content_daily_performance/month=2026-03/data_0.parquet")
dim_content = pd.read_parquet(f"{REPO}/dim_content.parquet")

print("fact (2026-03) shape:", fact.shape)
print("dim_content shape:", dim_content.shape)


fact (2026-03) shape: (9841378, 31)
dim_content shape: (519606, 26)


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

- **Feature** (knowable now, safe to use): `position_tier`, `word_count`, `content_type`,
  `main_intent`, `search_volume`.
- **Label / proxy:** `ctr_gap` (and the `ctr`, `tier_median_ctr` it's built from — never features).
- **Context** (grouping/joining only, never learned from): `client_hash_id`, `content_hash_id`.
- **Excluded, with why:**
  - `ctr` itself — the label is computed from it; used only to *demonstrate* the leak in Section 3,
    then removed.
  - `ga4_*`, `sessions_*`, `ai_*` columns — GA4 coverage is partial (`ga4_data_available` is not
    `True` for most clients this month) and unrelated to a GSC-CTR lane.
  - `gsc_avg_position` (the raw daily column) — replaced by a page-month `sum/sum` reconstruction
    to avoid averaging daily averages, a classic grain mistake.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

FEATURES = ["position_tier", "word_count", "content_type", "main_intent", "search_volume"]
LABEL = "ctr_gap"
CONTEXT = ["client_hash_id", "content_hash_id"]
EXCLUDED = ["ctr", "tier_median_ctr", "ga4_*/sessions_*/ai_* columns", "raw gsc_avg_position (daily)"]

print("features:", FEATURES)
print("label:", LABEL)
print("context:", CONTEXT)
print("excluded:", EXCLUDED)


features: ['position_tier', 'word_count', 'content_type', 'main_intent', 'search_volume']
label: ctr_gap
context: ['client_hash_id', 'content_hash_id']
excluded: ['ctr', 'tier_median_ctr', 'ga4_*/sessions_*/ai_* columns', 'raw gsc_avg_position (daily)']


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Three checks on the March partition — grain, counts/date span, and availability (filtered with
`IS TRUE`) — then a 5-feature frame with an "available when?" line each, then the deliberate leak.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# ---- Q1: GRAIN — one row = one report_date x client x content in the raw daily table? ----
dupe_check = fact.groupby(["report_date", "client_hash_id", "content_hash_id"]).size().reset_index(name="n")
dupes = dupe_check[dupe_check["n"] > 1]
print("Q1 grain check: duplicate groups =", len(dupes), "(expect 0)")

# ---- Q2: COUNTS + DATE SPAN ----
print("\nQ2 total rows:", len(fact))
print("distinct clients:", fact["client_hash_id"].nunique())
print("distinct content items:", fact["content_hash_id"].nunique())
print("date span:", fact["report_date"].min(), "to", fact["report_date"].max())

# ---- Q3: AVAILABILITY, filtered with IS TRUE ----
before = len(fact)
fact_avail = fact[fact["gsc_data_available"] == True].copy()
after = len(fact_avail)
print(f"\nQ3 gsc_data_available IS TRUE: {after:,} of {before:,} rows survive ({after/before:.1%})")


Q1 grain check: duplicate groups = 0 (expect 0)

Q2 total rows: 9841378
distinct clients: 55
distinct content items: 331437
date span: 2026-03-01 to 2026-03-31

Q3 gsc_data_available IS TRUE: 3,611,061 of 9,841,378 rows survive (36.7%)


In [14]:
# ---- Build the Lane 4 page-month slice from the available rows ----
agg = (fact_avail.groupby(["client_hash_id", "content_hash_id"])
       .agg(impressions=("gsc_impressions", "sum"),
            clicks=("gsc_clicks", "sum"),
            sum_position=("gsc_sum_position", "sum"))
       .reset_index())
agg["avg_position"] = agg["sum_position"] / agg["impressions"]
agg["ctr"] = agg["clicks"] / agg["impressions"] * 100
print("page-month rows before visibility filter:", agg.shape)

VISIBLE_MIN_IMPR = 150
visible = agg[(agg["impressions"] >= VISIBLE_MIN_IMPR) & (agg["avg_position"] > 0)].copy()
print("visible pages (impressions >=", VISIBLE_MIN_IMPR, "):", visible.shape)

bins, labels = [0, 3, 10, 20, 50, np.inf], ["top_3", "page_1", "striking", "page_3_5", "deep"]
visible["position_tier"] = pd.cut(visible["avg_position"], bins=bins, labels=labels)
print(visible["position_tier"].value_counts())

page-month rows before visibility filter: (176738, 7)
visible pages (impressions >= 150 ): (91974, 7)
position_tier
page_1      44463
striking    17631
page_3_5    17303
top_3        9733
deep         2844
Name: count, dtype: int64


In [15]:
visible["tier_median_ctr"] = visible.groupby("position_tier", observed=True)["ctr"].transform("median")
visible["ctr_gap"] = visible["ctr"] - visible["tier_median_ctr"]

content_cols = ["client_hash_id", "content_hash_id", "content_type", "main_intent", "word_count", "search_volume"]
visible = visible.merge(dim_content[content_cols], on=["client_hash_id", "content_hash_id"], how="left")
print("final slice:", visible.shape)
visible[["position_tier", "ctr", "tier_median_ctr", "ctr_gap", "content_type", "main_intent", "word_count", "search_volume"]].head()

final slice: (91974, 14)


,position_tier,ctr,tier_median_ctr,ctr_gap,content_type,main_intent,word_count,search_volume
0,striking,0.604230,0.124069,0.480160,keyword article,informational,3168.0,0.0
1,striking,0.000000,0.124069,-0.124069,keyword article,informational,3465.0,0.0
2,striking,0.000000,0.124069,-0.124069,keyword article,informational,3149.0,0.0
3,page_1,0.000000,0.201265,-0.201265,keyword article,transactional,4016.0,20.0
4,striking,0.645995,0.124069,0.521925,keyword article,informational,3804.0,20.0


**The 5 features — knowable at the decision moment because:**

1. **`position_tier`** — where the page currently ranks is known the instant you look it up; no
   future data needed.
2. **`word_count`** — a property of the published content itself, set before this month's traffic
   happened.
3. **`content_type`** — set at publish time (keyword article, comparison article, etc.), fixed
   regardless of later performance.
4. **`main_intent`** — assigned to the target keyword at creation time, not derived from outcomes.
5. **`search_volume`** — a keyword-level demand estimate that exists independently of how this
   specific page performed this month.

None of these are built from this month's `clicks`/`ctr` — that's the whole point of Section 4's trap.

In [16]:
FEATURES = ["position_tier", "word_count", "content_type", "main_intent", "search_volume"]
feat = visible[FEATURES + ["ctr_gap"]].dropna(subset=["ctr_gap"]).copy()
print("feature frame:", feat.shape)
feat.head()

feature frame: (91974, 6)


,position_tier,word_count,content_type,main_intent,search_volume,ctr_gap
0,striking,3168.0,keyword article,informational,0.0,0.480160
1,striking,3465.0,keyword article,informational,0.0,-0.124069
2,striking,3149.0,keyword article,informational,0.0,-0.124069
3,page_1,4016.0,keyword article,transactional,20.0,-0.201265
4,striking,3804.0,keyword article,informational,20.0,0.521925


**Now the trap.** Add `ctr` itself as a "feature" — it's the ingredient the label is built from —
and watch a quick linear-regression R² jump toward a suspiciously perfect number. Then delete it
and keep the honest score.

In [17]:
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split

def quick_score(feature_cols, df):
    d = df.dropna(subset=feature_cols + ["ctr_gap"]).copy()
    cat_cols = [c for c in feature_cols if str(d[c].dtype) in ("object", "category")]
    num_cols = [c for c in feature_cols if c not in cat_cols]
    pre = ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)], remainder="passthrough")
    pipe = Pipeline([("prep", pre), ("model", LinearRegression())])
    Xtr, Xte, ytr, yte = train_test_split(d[feature_cols], d["ctr_gap"], test_size=0.3, random_state=0)
    pipe.fit(Xtr, ytr)
    return r2_score(yte, pipe.predict(Xte))

honest_r2 = quick_score(FEATURES, feat)
print("honest R^2 (5 safe features):", round(honest_r2, 3))

# THE TRAP: add ctr, the ingredient of the label, on purpose
leaky_features = FEATURES + ["ctr"]
leaky_df = visible[leaky_features + ["ctr_gap"]].dropna()
leaky_r2 = quick_score(leaky_features, leaky_df)
print("leaky R^2 (adding `ctr`, which the label is built from):", round(leaky_r2, 3))

# ---- delete the leak, keep the honest number ----
del leaky_features  # the fix: never touch ctr as a feature
print("\nFinal, honest R^2 to report:", round(honest_r2, 3), "-- not", round(leaky_r2, 3))

honest R^2 (5 safe features): 0.004
leaky R^2 (adding `ctr`, which the label is built from): 1.0

Final, honest R^2 to report: 0.004 -- not 1.0


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Named limitation: this slice covers a minority of the panel, not the whole thing.** Only 55 of
104 clients (about 53%) have any row at all in the March 2026 partition, and only 36.7% of those
9.84M daily rows pass `gsc_data_available IS TRUE`. After the visibility filter (≥150 impressions,
real position), the final feature frame is 91,974 page-months — a meaningful but filtered slice,
skewed toward clients with active GSC access and pages that already get real traffic. Anything
learned here may not generalize to the ~half of clients with little or no usable search history
(consistent with the panel warning that "a third of clients have little or no usable search/
analytics history").

A second limitation worth naming: a few `search_volume` values in the sample above are exactly
`0.0` sitting next to nonzero impressions — worth checking whether that's a genuine zero-demand
keyword or a missing/unavailable code before trusting it as a feature at face value, per the
starter dataset's lesson that missingness tends to follow `content_type`, not randomness.

This is one calendar month only — no trend, no seasonality, and the tier-median CTRs are specific
to March 2026's SERP conditions (and to a `deep` tier bucket that's thin at only 2,844 rows).

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

print("clients present in March fact table:", fact["client_hash_id"].nunique(), "/ 104 total")
print("share of all clients represented:", round(fact["client_hash_id"].nunique() / 104, 3))

zero_volume_examples = visible[(visible["search_volume"] == 0) & (visible["impressions"] > 0)]
print("\npage-months with impressions>0 but search_volume==0:", len(zero_volume_examples),
      f"({len(zero_volume_examples)/len(visible):.1%} of the final slice)")

clients present in March fact table: 55 / 104 total
share of all clients represented: 0.529

page-months with impressions>0 but search_volume==0: 34459 (37.5% of the final slice)


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.